In [1]:
!pip install gtfparse
!pip install polars=='0.16.17'
!pip install pyarrow
!pip install anndata==0.8.0

In [2]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import pyarrow
from gtfparse import read_gtf
import scipy

In [3]:
SAM = sc.read_h5ad('../Active_SAM_joined/SAM_AC_ncbi_soupx_cleaned_03122025.h5ad')

In [5]:
np.sort(SAM.obs['key'].unique())

array(['Run12_sample1', 'Run12_sample10', 'Run12_sample11',
       'Run12_sample12', 'Run12_sample2', 'Run12_sample3',
       'Run12_sample5', 'Run12_sample6', 'Run12_sample7', 'Run12_sample9',
       'Run17_sample2', 'Run17_sample3', 'Run17_sample4'], dtype=object)

In [7]:
dat_list = ['AC/Outputs/Run12_sample2_ncbi_soupcorrected.csv',
'AC/Outputs/Run12_sample3_ncbi_soupcorrected.csv',
'AC/Outputs/Run12_sample5_ncbi_soupcorrected.csv',
           'AC/Outputs/Run12_sample6_ncbi_soupcorrected.csv',
           'AC/Outputs/Run12_sample7_ncbi_soupcorrected.csv',
           'AC/Outputs/Run12_sample9_ncbi_soupcorrected.csv',
           'AC/Outputs/Run12_sample10_ncbi_soupcorrected.csv',
           'AC/Outputs/Run12_sample11_ncbi_soupcorrected.csv',
           'AC/Outputs/Run12_sample12_ncbi_soupcorrected.csv',
           'AC/Outputs/Run17_sample2_ncbi_soupcorrected.csv',
           'AC/Outputs/Run17_sample3_ncbi_soupcorrected.csv',
           'AC/Outputs/Run17_sample4_ncbi_soupcorrected.csv',]

In [8]:
tot_dat = ad.read_csv('AC/Outputs/Run12_sample1_ncbi_soupcorrected.csv',)

tot_dat = tot_dat.T
tot_dat.X = scipy.sparse.csr_matrix(tot_dat.X)
counts = np.sum(tot_dat.X, axis = 1).A.reshape((1,len(tot_dat)))[0]
genes = np.count_nonzero(tot_dat.X.A, axis = 1)

tot_dat.obs['n_counts'] = counts
tot_dat.obs['n_genes'] = genes
tot_dat.obs['key'] = 'Run12_sample1'
tot_dat.var_names = [i for i in tot_dat.var_names]

for dn in dat_list:
    dat = ad.read_csv(dn)
    dat = dat.T
    dat.X = scipy.sparse.csr_matrix(dat.X)
    counts = np.sum(dat.X, axis = 1).A.reshape((1,len(dat)))[0]
    genes = np.count_nonzero(dat.X.A, axis = 1)
    
    dat.obs['n_counts'] = counts
    dat.obs['n_genes'] = genes
    dat.obs['key'] = dn.split('_')[1] + '_' + dn.split('_')[2]
    dat.var_names = [i for i in dat.var_names]
    
    tot_dat = ad.concat([tot_dat, dat], join = 'outer')
    print(tot_dat.shape)

/scratch/miniconda/lib/python3.7/site-packages/anndata/_core/anndata.py:1828: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


(5796, 22251)
(8471, 22445)
(11061, 22633)
(13673, 22753)
(15714, 22796)
(18531, 22851)
(20895, 22902)
(24129, 22934)
(27043, 22957)
(33551, 23072)
(40169, 23172)
(50953, 23671)


In [19]:
tot_dat.var_names

Index(['LOC100337544', 'LOC100337546', 'LOC100551521', 'LOC100551532',
       'LOC100551533', 'LOC100551546', 'LOC100551547', 'LOC100551550',
       'LOC100551552', 'LOC100551560',
       ...
       'zswim7', 'zswim8', 'zswim9', 'zup1', 'zw10', 'zwilch', 'zxdc', 'zyx',
       'zzef1', 'zzz3'],
      dtype='object', length=23671)

In [10]:
db_gene = read_gtf('AC/Anolis_carolinensis.AnoCar2.0v2.112.gtf', features = ['gene', 'transcript'])
df_gene = db_gene.to_pandas()

INFO:root:Extracted GTF attributes: ['gene_id', 'gene_version', 'gene_name', 'gene_source', 'gene_biotype', 'transcript_id', 'transcript_version', 'transcript_name', 'transcript_source', 'transcript_biotype', 'tag', 'projection_parent_transcript']


In [11]:
df_gene

,seqname,source,feature,start,end,score,strand,frame,gene_id,gene_version,gene_name,gene_source,gene_biotype,transcript_id,transcript_version,transcript_name,transcript_source,transcript_biotype,tag,projection_parent_transcript
0,1,ensembl,gene,79277,182777,NaN,-,0,ENSACAG00000009394,4,JAG2,ensembl,protein_coding,,,,,,,
1,1,ensembl,transcript,79277,182777,NaN,-,0,ENSACAG00000009394,4,JAG2,ensembl,protein_coding,ENSACAT00000009589,4,JAG2-201,ensembl,protein_coding,Ensembl_canonical,
2,1,ensembl,transcript,79277,182777,NaN,-,0,ENSACAG00000009394,4,JAG2,ensembl,protein_coding,ENSACAT00000043269,1,JAG2-202,ensembl,protein_coding,,
3,1,ensembl,gene,31575205,31576551,NaN,-,0,ENSACAG00000037302,1,,ensembl,protein_coding,,,,,,,
4,1,ensembl,transcript,31575205,31576551,NaN,-,0,ENSACAG00000037302,1,,ensembl,protein_coding,ENSACAT00000037515,1,,ensembl,protein_coding,Ensembl_canonical,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65031,AAWZ02041975.1,ensembl,transcript,21,4996,NaN,-,0,ENSACAG00000044031,1,,ensembl,protein_coding,ENSACAT00000037660,1,,ensembl,protein_coding,Ensembl_canonical,
65032,AAWZ02041982.1,ensembl,gene,21,3556,NaN,+,0,ENSACAG00000041671,1,,ensembl,protein_coding,,,,,,,
65033,AAWZ02041982.1,ensembl,transcript,21,3556,NaN,+,0,ENSACAG00000041671,1,,ensembl,protein_coding,ENSACAT00000047420,1,,ensembl,protein_coding,Ensembl_canonical,
65034,AAWZ02041985.1,ensembl,gene,1105,3771,NaN,+,0,ENSACAG00000041116,1,,ensembl,protein_coding,,,,,,,


In [12]:
fin_gene_name = {}
for item in tot_dat.var_names:
    gn = df_gene.loc[df_gene.index[df_gene['gene_id'] == item][0], 'gene_name']
    if gn != "":
        fin_gene_name[item] = gn
    else:
        fin_gene_name[item] = item

IndexError: index 0 is out of bounds for axis 0 with size 0

In [21]:
mapping = pd.read_csv('../BLASTMAPPING/maps/active_maps/hypo_proj/mgac/mg_to_ac.txt', delimiter = '\t', header = None)

In [22]:
tot_dat.var_names = [fin_gene_name[i] for i in tot_dat.var_names]

KeyError: 'LOC100337544'

In [23]:
a = 0
mo_set = set(mapping[1].unique())
for item in tot_dat.var_names:
    if item in mo_set:
        a += 1
a

18742

In [24]:
tot_dat.var_names

Index(['LOC100337544', 'LOC100337546', 'LOC100551521', 'LOC100551532',
       'LOC100551533', 'LOC100551546', 'LOC100551547', 'LOC100551550',
       'LOC100551552', 'LOC100551560',
       ...
       'zswim7', 'zswim8', 'zswim9', 'zup1', 'zw10', 'zwilch', 'zxdc', 'zyx',
       'zzef1', 'zzz3'],
      dtype='object', length=23671)

In [25]:
tot_dat.obs_names_make_unique()
tot_dat.var_names_make_unique()

In [26]:
tot_dat.obs['key'].unique()

array(['Run12_sample1', 'sample2_ncbi', 'sample3_ncbi', 'sample5_ncbi',
       'sample6_ncbi', 'sample7_ncbi', 'sample9_ncbi', 'sample10_ncbi',
       'sample11_ncbi', 'sample12_ncbi', 'sample4_ncbi'], dtype=object)

In [13]:
test_dat = sc.read_h5ad('../Testing_Raw_Dat_RNASEQ_Joined/Backupable/tot_dat_AC_ncbi_soupx_cleaned_03122025.h5ad')

In [30]:
len(set(test_dat.obs_names) & set(tot_dat.obs_names))/len(test_dat)

1.0

In [31]:
fin_obs = set(test_dat.obs_names) & set(tot_dat.obs_names)

In [32]:
sub_test_dat = test_dat[test_dat.obs_names.isin(fin_obs)]
sub_tot_dat = tot_dat[tot_dat.obs_names.isin(fin_obs)]

In [33]:
sub_test_dat.X

<47900x23671 sparse matrix of type '<class 'numpy.float32'>'
	with 73086009 stored elements in Compressed Sparse Column format>

In [34]:
sub_tot_dat.X

<47900x23671 sparse matrix of type '<class 'numpy.float32'>'
	with 73086009 stored elements in Compressed Sparse Row format>

In [12]:
tot_dat.write('../Testing_Raw_Dat_RNASEQ_Joined/tot_dat_CT_Run21_Run16s5s7_01242026.h5ad')